# Fee Rate — Bybit

Fetches **maker / taker fee rates** for one or more symbols via `GET /v5/account/fee-rate`.

- Supports `spot`, `linear`, `inverse`, and `option` categories.
- Leave `SYMBOLS` empty to fetch all symbols for the given category.
- **Always uses mainnet** — Bybit's demo environment does not support this endpoint.
- Reads `BYBIT_MAINNET_API_KEY` / `BYBIT_MAINNET_API_SECRET` from `.env` first;  
  falls back to `BYBIT_API_KEY` / `BYBIT_API_SECRET`.

In [ ]:
# SECTION 1 — Imports & config
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

try:
    from dotenv import load_dotenv
    _env = _ROOT / ".env"
    if _env.exists():
        load_dotenv(_env)
        print(f"Loaded .env from {_env}")
    else:
        print(f"No .env at {_env}")
except ImportError:
    print("python-dotenv not installed")

# ── Parameters ──────────────────────────────────────────────────────────────
CATEGORY = "linear"          # spot | linear | inverse | option

# Symbols to query. Leave empty to fetch all symbols for the category.
SYMBOLS  = ["BTCUSDT", "ETHUSDT", "XAUUSDT"]

# fee-rate is mainnet-only — demo/testnet are forced off
print(f"Endpoint: mainnet  |  category: {CATEGORY}  |  symbols: {SYMBOLS or 'ALL'}")

In [ ]:
# SECTION 2 — Bybit mainnet client
# /v5/account/fee-rate is not available on demo or testnet.
from pybit.unified_trading import HTTP

# api_key    = os.getenv("BYBIT_MAINNET_API_KEY")    or os.getenv("BYBIT_API_KEY",    "")
# api_secret = os.getenv("BYBIT_MAINNET_API_SECRET") or os.getenv("BYBIT_API_SECRET", "")
api_key    = "7bruM7c1bbLwZa7POt"
api_secret = "PzaXHSUassodHWaQ0iwhmdBp7ML9OUF2wWkh"

if not api_key:
    raise RuntimeError(
        "No API key found. Set BYBIT_MAINNET_API_KEY (or BYBIT_API_KEY) in your .env file."
    )

client = HTTP(
    testnet=False,
    demo=False,
    api_key=api_key,
    api_secret=api_secret,
)

print(f"HTTP mainnet  |  API key: {api_key[:6]}...{api_key[-4:]}")

In [ ]:
# SECTION 3 — Fetch fee rates

def fetch_fee_rates(category: str, symbols: list[str]) -> pd.DataFrame:
    rows: list[dict] = []

    if symbols:
        for sym in symbols:
            resp = client.get_fee_rates(category=category, symbol=sym)
            if resp.get("retCode") != 0:
                raise RuntimeError(
                    f"Bybit error {resp.get('retCode')}: {resp.get('retMsg')}  (symbol={sym})"
                )
            for item in (resp.get("result") or {}).get("list") or []:
                item.setdefault("category", category)
                rows.append(item)
    else:
        resp = client.get_fee_rates(category=category)
        if resp.get("retCode") != 0:
            raise RuntimeError(
                f"Bybit error {resp.get('retCode')}: {resp.get('retMsg')}"
            )
        for item in (resp.get("result") or {}).get("list") or []:
            item.setdefault("category", category)
            rows.append(item)

    if not rows:
        return pd.DataFrame(columns=["category", "symbol", "makerFeeRate", "takerFeeRate"])

    df = pd.DataFrame(rows)
    for col in ("makerFeeRate", "takerFeeRate"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    ordered = [c for c in ["category", "symbol", "baseCoin", "makerFeeRate", "takerFeeRate"] if c in df.columns]
    return df[ordered + [c for c in df.columns if c not in ordered]].reset_index(drop=True)


fee_df = fetch_fee_rates(CATEGORY, SYMBOLS)
print(f"Fetched {len(fee_df)} row(s)  |  category: {CATEGORY}")

In [ ]:
# SECTION 4 — Display results

if fee_df.empty:
    print("No fee-rate data returned.")
else:
    display_df = fee_df.copy()
    for col in ("makerFeeRate", "takerFeeRate"):
        if col in display_df.columns:
            display_df[col] = display_df[col].map(
                lambda x: f"{x * 100:.4f}%" if pd.notna(x) else ""
            )
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
        display(display_df)